In [4]:
import os
import re
import sys
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)


warnings.filterwarnings("ignore")
CSV_PATH = os.environ.get("LOAN_CSV", "dataset/SingleCycleLoaneeMember.csv")

d = pd.read_csv(CSV_PATH)


In [3]:
a=d[d['loan_amount'] >= 300000].copy()
a['loan_Component'].value_counts()

loan_Component
AGROSOR          10405
AGROSOR-RAISE       92
AGROSOR-MFCE        75
Name: count, dtype: int64

In [5]:
# from basemodel_notebook
LADDER = np.array([5, 10, 20, 25, 30, 40, 50, 60, 70, 80, 90, 100, 120, 150,
                   200, 250, 300, 400, 500, 800, 1000, 2000, 5000]) * 1000

# Feature sets, smallest to largest. The ablation table shows what each addition buys.
FEATURE_SETS = {
    "1 income": ["monthly_income"],
    "2 +purpose": ["monthly_income", "purpose"],
    "3 +land": ["monthly_income", "purpose", "land"],
    "4 +family size": ["monthly_income", "purpose", "land", "family_size"],
    "5 +earners,expenses": ["monthly_income", "purpose", "land", "family_size",
                            "earners", "yearly_expenses"],
    "6 +district": ["monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district"],
    "7 +gender": [ "monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district", "gender"],
    "8 +occupation": [ "monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district", "gender", "occupation" ],
    "9 +home_owner_type": [ "monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district", "gender", "occupation", "home_owner_type" ],
    "10 +loan_Component": [ "monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district", "gender", "occupation", "home_owner_type", "loan_Component"],
    "11 +year_of_rent": [ "monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district", "gender", "occupation", "home_owner_type", "loan_Component",
                          "year_of_rent"],
    "12 +installmenttype": [ "monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district", "gender", "occupation", "home_owner_type", "loan_Component",
                          "year_of_rent", "installmenttype"],
    "13 +installmentyear": [ "monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district", "gender", "occupation", "home_owner_type", "loan_Component",
                          "year_of_rent", "installmenttype", "installmentyear"],
    "14 +totalInstallmentcount": [ "monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district", "gender", "occupation", "home_owner_type", "loan_Component",
                          "year_of_rent", "installmenttype", "installmentyear", "totalInstallmentcount"],
    "15 +maritial_status": [ "monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district", "gender", "occupation", "home_owner_type", "loan_Component",
                          "year_of_rent", "installmenttype", "installmentyear", "totalInstallmentcount", "maritial_status"],
    "16 +education": [ "monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district", "gender", "occupation", "home_owner_type", "loan_Component",
                          "year_of_rent", "installmenttype", "installmentyear", "totalInstallmentcount", "maritial_status", "education"],
    "17 +alternative_occupation": [ "monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district", "gender", "occupation", "home_owner_type", "loan_Component",
                          "year_of_rent", "installmenttype", "installmentyear", "totalInstallmentcount", "maritial_status", "education", "alternative_occupation"],
    "18 +admission_age": [ "monthly_income", "purpose", "land", "family_size",
                    "earners", "yearly_expenses", "district", "gender", "occupation", "home_owner_type", "loan_Component",
                    "year_of_rent", "installmenttype", "installmentyear", "totalInstallmentcount", "maritial_status",
                       "education", "alternative_occupation", "admission_age"],
}
DEFAULT_SET = "18 +admission_age"
CAT_FEATURES = ["purpose", "district", 'gender','occupation','home_owner_type', 'loan_Component', 'installmenttype', 'maritial_status', 'education', 'alternative_occupation']

def clean_purpose(s: pd.Series) -> pd.Series:
    """'Dairy Farm (ডেইরি ফার্ম)' -> 'Dairy Farm'"""
    return s.fillna("NA").astype(str).str.replace(r"\s*\(.*$", "", regex=True).str.strip()


def fit_meta(train_df: pd.DataFrame) -> dict:
    purpose = clean_purpose(train_df["loan_use_sources"])
    counts = purpose.value_counts()

    def categories(column):
        values = (
            train_df[column]
            .astype("string")
            .str.strip()
            .dropna()
            .unique()
            .tolist()
        )
        return sorted(values)

    return {
        "income_cap": float(
            train_df["total_monthly_income"].quantile(0.999)
        ),
        "expense_cap": float(
            train_df["yearly_total_expenses"].quantile(0.999)
        ),
        "land_sentinel": 1e5,
        "purposes": sorted(counts[counts >= 100].index.tolist()),
        "districts": categories("current_addr_district"),
        "gender": categories("gender"),
        "occupation": categories("occupation"),
        "home_owner_type": categories("home_owner_type"),
        "loan_Component": categories("loan_Component"),
        "installmenttype": categories("installmenttype"),
        "maritial_status": categories("maritial_status"),
        "education": categories("education"),
        "alternative_occupation": categories("alternative_occupation")
    }

def build_features(df: pd.DataFrame, meta: dict, features=None) -> pd.DataFrame:
    """Build only the requested features, so unused input columns are not required."""
    builders = {
        "monthly_income": lambda: df["total_monthly_income"].clip(upper=meta["income_cap"]),
        "purpose": lambda: pd.Categorical(
            (lambda p: p.where(p.isin(meta["purposes"]), "rare"))(clean_purpose(df["loan_use_sources"])),
            categories=meta["purposes"] + ["rare"]),                    # unseen/rare -> 'rare'
        "land": lambda: df["farm_land_amt"].where(df["farm_land_amt"] < meta["land_sentinel"]),
        "family_size": lambda: df["total_family_member"].clip(upper=12),
        "earners": lambda: df["total_earning_member"],
        'year_of_rent' : lambda: df["year_of_rent"],
        'installmentyear': lambda: df["installmentyear"],
        'totalInstallmentcount': lambda: df["totalInstallmentcount"],
        'admission_age' : lambda: (pd.to_datetime(df['admission_date']) - pd.to_datetime(df["dateofbirth"])).dt.days//365,
        "yearly_expenses": lambda: df["yearly_total_expenses"].clip(upper=meta["expense_cap"]),
        "district": lambda: pd.Categorical(
            (lambda d: d.where(d.isin(meta["districts"])))(df["current_addr_district"].astype(str).str.strip()),
            categories=meta["districts"]),  # unseen -> missing
        'gender': lambda: pd.Categorical(
            (lambda d: d.where(d.isin(meta["gender"])))(df['gender'].astype(str))
        ),
        'occupation': lambda: pd.Categorical(
            (lambda d: d.where(d.isin(meta["occupation"])))(df['occupation'].astype(str))
        ),
        'home_owner_type': lambda: pd.Categorical(
            (lambda d: d.where(d.isin(meta["home_owner_type"])))(df['home_owner_type'].astype(str))
        ),
        'loan_Component': lambda: pd.Categorical(
            (lambda d: d.where(d.isin(meta["loan_Component"])))(df['loan_Component'].astype(str))
        ),
        'maritial_status': lambda: pd.Categorical(
            (lambda d: d.where(d.isin(meta["maritial_status"])))(df['maritial_status'].astype(str))
        ),
        'installmenttype': lambda: pd.Categorical(
            (lambda d: d.where(d.isin(meta["installmenttype"])))(df['installmenttype'].astype(str))
        ),
        'education': lambda: pd.Categorical(
            (lambda d: d.where(d.isin(meta["education"])))(df['education'].astype(str))
        ),
        'alternative_occupation' : lambda: pd.Categorical(
            (lambda d: d.where(d.isin(meta["alternative_occupation"])))(df['alternative_occupation'].astype(str))
        ),

    }
    X = pd.DataFrame(index=df.index)
    for f in (features or list(builders)):
        X[f] = builders[f]()
    return X

def rung(v):
    return np.abs(np.log(np.asarray(v, float))[:, None] - np.log(LADDER)[None, :]).argmin(1)


def metrics(actual, pred) -> dict:
    a = np.asarray(actual, float)
    p = np.clip(np.asarray(pred, float), 4000, 5e6)
    la, lp = np.log1p(a), np.log1p(p)
    big = a >= 300000
    return {
        "MAE": float(np.mean(np.abs(p - a))),
        "MedianAE": float(np.median(np.abs(p - a))),
        "RMSLE": float(np.sqrt(np.mean((lp - la) ** 2))),
        "R2_log": float(1 - np.sum((la - lp) ** 2) / np.sum((la - la.mean()) ** 2)),
        "within_1_ladder_step": float(np.mean(np.abs(rung(p) - rung(a)) <= 1)),
        "exact_ladder_step": float(np.mean(rung(p) == rung(a))),
        "AUC_loan_ge_300k": float(roc_auc_score(big, np.log(p))) if 0 < big.sum() < len(a) else float("nan"),
    }

In [6]:
# Same time-ordered 80/20 split as the base model
df = pd.read_csv(CSV_PATH)
df["_adm"] = pd.to_datetime(df["admission_date"])
df = df.sort_values(["_adm", "member_id"]).reset_index(drop=True)
y = df["loan_amount"].to_numpy(float)
cut = int(len(df) * 0.8)
tr, te = df.iloc[:cut], df.iloc[cut:]

meta = fit_meta(tr)                                   # constants learned from TRAIN only
Xtr, Xte = build_features(tr, meta), build_features(te, meta)
F18 = FEATURE_SETS["18 +admission_age"]                   # the base model's features
F18 = FEATURE_SETS["18 +admission_age"]                      # + earners, expenses, district

In [19]:
def make_regressor(seed=0, **kw):
    """Same settings as the base model, but learning_rate 0.1 so experiments run faster."""
    p = dict(objective="l1", learning_rate=0.03, num_leaves=15, min_child_samples=100,
             subsample=0.8, subsample_freq=1, reg_lambda=5, cat_smooth=30,
             min_data_per_group=50, cat_l2=10, n_estimators=3000, verbose=-1, random_state=seed)
    p.update(kw)
    return lgb.LGBMRegressor(**p)

def fit_reg(X, y_log, seed=0, **kw):
    """Fit on the first 85% of rows, early-stop on the most recent 15%."""
    k = int(len(X) * 0.85)
    m = make_regressor(seed, **kw)
    m.fit(X.iloc[:k], y_log[:k], eval_set=[(X.iloc[k:], y_log[k:])], eval_metric="l1",
          callbacks=[lgb.early_stopping(60, verbose=False)])
    return m

def short(r: dict) -> dict:
    keep = ("MAE", "RMSLE", "R2_log", "within_1_ladder_step", "exact_ladder_step", "AUC_loan_ge_300k")
    return {k: round(float(v), 3) for k, v in r.items() if k in keep}

5

### Experiment-  1

In [22]:
rows = []
for name, feats in [("18 features", F18), ("18 features (+admission_age)", F18)]:
    for frac, n_seeds in [(0.05, 3), (0.10, 3), (0.25, 2), (0.50, 1), (1.00, 1)]:
        n = int(cut * frac)
        results = []
        for s in range(n_seeds):
            # random subset, kept in time order so early stopping still uses the newest rows
            idx = (np.sort(np.random.RandomState(100 + s).choice(cut, n, replace=False))
                   if frac < 1 else np.arange(cut))
            m = fit_reg(Xtr.iloc[idx][feats], np.log1p(y[idx]), seed=s)
            results.append(metrics(y[cut:], np.expm1(m.predict(Xte[feats]))))
        r = pd.DataFrame(results)
        rows.append([name, n, r.MAE.mean(), r.MAE.std() if n_seeds > 1 else np.nan,
                     r.R2_log.mean(), r.within_1_ladder_step.mean()])
        print(rows[-1], flush=True)

table = pd.DataFrame(rows, columns=["features", "train_rows", "MAE", "MAE_sd", "R2_log", "within1"])
print("\n", table.round(3).to_string(index=False))
table.to_csv("learning_curve.csv", index=False)

['18 features', 4959, np.float64(32089.95414044903), np.float64(472.8251860119409), np.float64(0.6728107626738096), np.float64(0.5497667916717072)]
['18 features', 9919, np.float64(30501.226367469233), np.float64(176.7524190436226), np.float64(0.6855005617819206), np.float64(0.5627646276059518)]
['18 features', 24798, np.float64(29646.85598845475), np.float64(73.12322855780927), np.float64(0.6980656457240226), np.float64(0.5685713133594097)]
['18 features', 49597, np.float64(28890.605636471268), nan, np.float64(0.7082518960076447), np.float64(0.5794185249405218)]
['18 features', 99194, np.float64(28466.88183077994), nan, np.float64(0.7130822456770654), np.float64(0.5844187265615549)]
['18 features (+admission_age)', 4959, np.float64(32089.95414044903), np.float64(472.8251860119409), np.float64(0.6728107626738096), np.float64(0.5497667916717072)]
['18 features (+admission_age)', 9919, np.float64(30501.226367469233), np.float64(176.7524190436226), np.float64(0.6855005617819206), np.float

### Experiment- 2
- experiment demonstrates that predicting loan amount as a probability distribution over discrete loan amount classes can outperform direct regression

In [12]:
# (a) how many rows per amount?
counts = pd.Series(y[:cut]).value_counts()
print("distinct loan amounts (all rows):", len(np.unique(y)), "| in train:", len(counts))
for t in [5000, 1000, 300, 100, 30, 10]:
    print(f"amounts with >= {t:>4} rows in train: {(counts >= t).sum():>3}"
          f"  (cover {counts[counts >= t].sum() / cut:.1%} of loans)")

# (b) multiclass over the ladder
classes = np.sort(counts[counts >= 300].index.values)
K = len(classes)
print("classes used:", K, classes.astype(int).tolist())
nearest = lambda v: classes[np.abs(np.log(v)[:, None] - np.log(classes)[None, :]).argmin(1)]
y_cls = np.searchsorted(classes, nearest(y[:cut]))           # class index of each training loan

k = int(cut * 0.85)
clf = lgb.LGBMClassifier(objective="multiclass", num_class=K, learning_rate=0.1, num_leaves=15,
                         min_child_samples=100, subsample=0.8, subsample_freq=1, reg_lambda=5,
                         cat_smooth=30, min_data_per_group=50, cat_l2=10, n_estimators=800,
                         verbose=-1, random_state=0)
clf.fit(Xtr.iloc[:k][F18], y_cls[:k], eval_set=[(Xtr.iloc[k:][F18], y_cls[k:])],
        callbacks=[lgb.early_stopping(30, verbose=False)])
P = clf.predict_proba(Xte[F18])                               # probability of each amount class

# four ways to turn a probability distribution into one number
arg = classes[P.argmax(1)]                                   # most likely amount
median = classes[(P.cumsum(1) >= 0.5).argmax(1)]             # median of the distribution
geo = np.exp((P * np.log(classes)[None, :]).sum(1))          # geometric mean
mean = (P * classes[None, :]).sum(1)                         # expected value

reg = fit_reg(Xtr[F18], np.log1p(y[:cut]))
reg_pred = np.expm1(reg.predict(Xte[F18]))
a = y[cut:]
R = {"classifier: argmax (most likely amount)": metrics(a, arg),
     "classifier: median of predicted distribution": metrics(a, median),
     "classifier: geometric mean": metrics(a, geo),
     "classifier: expected value": metrics(a, mean),
     "regression (base model)": metrics(a, reg_pred)}
print("\n", pd.DataFrame({k_: short(v) for k_, v in R.items()}).T.round(3).to_string())

big = a >= 300000
for nm, p in [("argmax", arg), ("median-of-dist", median), ("regression", reg_pred)]:
    print(f"{nm:15s} MAE normal {np.mean(np.abs(p - a)[~big]):8.0f} | MAE big {np.mean(np.abs(p - a)[big]):8.0f}"
          f" | median pred for big loans {np.median(p[big]):8.0f} | for <=30k loans {np.median(p[a <= 30000]):8.0f}")

distinct loan amounts (all rows): 120 | in train: 116
amounts with >= 5000 rows in train:   8  (cover 78.1% of loans)
amounts with >= 1000 rows in train:  13  (cover 94.0% of loans)
amounts with >=  300 rows in train:  21  (cover 97.5% of loans)
amounts with >=  100 rows in train:  27  (cover 98.5% of loans)
amounts with >=   30 rows in train:  44  (cover 99.5% of loans)
amounts with >=   10 rows in train:  61  (cover 99.8% of loans)
classes used: 21 [10000, 20000, 25000, 30000, 40000, 45000, 48000, 50000, 60000, 70000, 80000, 90000, 95000, 100000, 120000, 150000, 200000, 300000, 400000, 500000, 800000]

                                                     MAE  RMSLE  R2_log  within_1_ladder_step  exact_ladder_step  AUC_loan_ge_300k
classifier: argmax (most likely amount)       32911.529  0.488   0.555                 0.525              0.331             0.999
classifier: median of predicted distribution  29478.810  0.431   0.653                 0.535              0.289             0.9

### Experiment- 3
- sample with component
- different component
- find probality of big loan amount
- Then train two regressor model one for normal loan and other for big loan
- Lastly understand the result
-  A member have applied on aggrosor can we loaned him on Jagoron with low amount ( or remove component)

In [13]:
a = y[cut:]
big_te = a >= 300000
ly = np.log1p(y)


def component_group(d):
    return np.where(d.loan_Component.str.startswith("AGROSOR"), "AGROSOR",
                    np.where(d.loan_Component.isin(["JAGORON", "SUFOLON"]), d.loan_Component, "other"))


g_tr, g_te = component_group(tr), component_group(te)
R = {}

# A. honest baseline
m = fit_reg(Xtr[F18], ly[:cut])
base = np.expm1(m.predict(Xte[F18]))
R["A. single model, 10 features (honest baseline)"] = metrics(a, base)

# B. component as an extra feature (LEAKY)
cats = sorted(set(g_tr))
Xc_tr, Xc_te = Xtr[F18].copy(), Xte[F18].copy()
Xc_tr["comp"] = pd.Categorical(g_tr, categories=cats)
Xc_te["comp"] = pd.Categorical(g_te, categories=cats)
mc = fit_reg(Xc_tr, ly[:cut])
R["B. single model + component as feature (LEAKY)"] = metrics(a, np.expm1(mc.predict(Xc_te)))

# C. one model per component, component known at prediction time (LEAKY)
pred_c = np.zeros(len(te))
for g in cats:
    i, j = np.where(g_tr == g)[0], np.where(g_te == g)[0]
    mg = fit_reg(Xtr.iloc[i][F18], ly[i], min_child_samples=min(100, max(20, len(i) // 50)))
    pred_c[j] = np.expm1(mg.predict(Xte.iloc[j][F18]))
    print(g, "train rows", len(i), "test rows", len(j), flush=True)
R["C. separate model per component, component known (LEAKY)"] = metrics(a, pred_c)

# D. legitimate two-stage: P(big) from features, then route to a normal or big regressor
is_big = y[:cut] >= 300000
k = int(cut * 0.85)
clf = lgb.LGBMClassifier(objective="binary", learning_rate=0.1, num_leaves=15, min_child_samples=100,
                         subsample=0.8, subsample_freq=1, reg_lambda=5, cat_smooth=30,
                         min_data_per_group=50, cat_l2=10, n_estimators=800, verbose=-1, random_state=0)
clf.fit(Xtr.iloc[:k][F18], is_big[:k], eval_set=[(Xtr.iloc[k:][F18], is_big[k:])],
        callbacks=[lgb.early_stopping(30, verbose=False)])
p = clf.predict_proba(Xte[F18])[:, 1]
m_normal = fit_reg(Xtr[~is_big][F18], ly[:cut][~is_big])
m_big = fit_reg(Xtr[is_big][F18], ly[:cut][is_big], min_child_samples=40)
log_n, log_b = m_normal.predict(Xte[F18]), m_big.predict(Xte[F18])

# Understand the result
R["D1. two-stage, hard route at P(big)>0.5"] = metrics(a, np.expm1(np.where(p > 0.5, log_b, log_n)))
R["D2. two-stage, hard route at P(big)>0.35"] = metrics(a, np.expm1(np.where(p > 0.35, log_b, log_n)))
R["D3. two-stage, soft blend (log scale)"] = metrics(a, np.expm1(p * log_b + (1 - p) * log_n))
#Instead of selecting only one regression model, it combines both predictions

print("\n", pd.DataFrame({k_: short(v) for k_, v in R.items()}).T.round(3).to_string())
for nm, pr in [("A single", base), ("C per-component (leaky)", pred_c),
               ("D1 two-stage", np.expm1(np.where(p > 0.5, log_b, log_n))),
               ("D3 soft", np.expm1(p * log_b + (1 - p) * log_n))]:
    print(f"{nm:26s} MAE normal {np.mean(np.abs(pr - a)[~big_te]):9.0f} | MAE big {np.mean(np.abs(pr - a)[big_te]):9.0f}"
          f" | median pred for big {np.median(pr[big_te]):9.0f}")
print("AUC of P(big):", round(roc_auc_score(big_te, p), 3),
      "| big loans with P>0.5:", round((p[big_te] > 0.5).mean(), 3),
      "| precision at P>0.5:", round(big_te[p > 0.5].mean(), 3),
      "| non-big flagged:", round((p[~big_te] > 0.5).mean(), 4))

AGROSOR train rows 8829 test rows 2229
JAGORON train rows 72291 test rows 17987
SUFOLON train rows 17082 test rows 4367
other train rows 992 test rows 216

                                                                 MAE  RMSLE  R2_log  within_1_ladder_step  exact_ladder_step  AUC_loan_ge_300k
A. single model, 4 features (honest baseline)             32217.579  0.430   0.655                 0.524              0.243             0.998
B. single model + component as feature (LEAKY)            32218.520  0.430   0.654                 0.525              0.243             0.998
C. separate model per component, component known (LEAKY)  29995.244  0.421   0.670                 0.540              0.246             0.999
D1. two-stage, hard route at P(big)>0.5                   29859.181  0.419   0.673                 0.539              0.245             0.999
D2. two-stage, hard route at P(big)>0.35                  29847.910  0.419   0.673                 0.539              0.245          

### Experiment- 4

In [17]:
purpose = clean_purpose(df["loan_use_sources"]).str.lower()

def purpose_family(s: str) -> str:
    if re.search(r"immigrant|abroad", s):
        return "Migration"
    if re.search(r"dairy|goru|cattle fatt|cow|goat|poultry|duck|sheep|livestock|animal|motataja|rearing|pigeon|farm(ing)?$", s):
        return "Livestock/Crop/Transport"
    if re.search(r"fish|shrimp|pond", s):
        return "Trade/Housing/Fishery"
    if re.search(r"paddy|vegetable cult|potato|jute|onion|betel|corn|maize|cultiv|wheat|mustard|chili|garlic|banana|mango|tobacco|crop|nursery|agricultur|seed", s):
        return "Livestock/Crop/Transport"
    if re.search(r"house|home|building|repair|toilet|latrine|tube|solar|sanitation|water", s):
        return "Trade/Housing/Fishery"
    if re.search(r"auto|vehicle|rickshaw|van|transport|truck|cng|bus$|tempo|motor|bike|boat|driver|easy bike|battery", s):
        return "Livestock/Crop/Transport"
    if re.search(r"business|shop|store|trade|dokan|stall|tea|grocery|cloth|garment|hardware|restaurant|hotel|medicine|pharmacy|distributor|mobile|electric|jewel|crockery|sanitary|brick|saw|mill|tailor|cosmetic|furniture|fertilizer|feed|wholesale|retail|bakery|food|fast|sweet|salon|parlor|barber|gas|plastic|glass", s):
        return "Trade/Housing/Fishery"
    return "Other"


family = purpose.map(purpose_family).values
print(pd.Series(family).value_counts(normalize=True).round(3).to_dict())

ly = np.log1p(y)
a = y[cut:]
single = np.expm1(fit_reg(Xtr[F18], ly[:cut]).predict(Xte[F18]))

separate = np.zeros(len(te))
for g in np.unique(family):
    i, j = np.where(family[:cut] == g)[0], np.where(family[cut:] == g)[0]
    mg = fit_reg(Xtr.iloc[i][F18], ly[i], min_child_samples=min(100, max(20, len(i) // 50)))   # define and train model
    separate[j] = np.expm1(mg.predict(Xte.iloc[j][F18]))
    print(g, "train rows", len(i), "test rows", len(j), flush=True)

R = {"single model (4 features)": metrics(a, single),
     "separate model per purpose family": metrics(a, separate)}
print("\n", pd.DataFrame({k: short(v) for k, v in R.items()}).T.round(3).to_string())
for g in np.unique(family):
    j = np.where(family[cut:] == g)[0]
    print(f"{g:26s} n={len(j):5d}  MAE single {np.mean(np.abs(single - a)[j]):9.0f}"
          f"  separate {np.mean(np.abs(separate - a)[j]):9.0f}")

{'Livestock/Crop/Transport': 0.684, 'Trade/Housing/Fishery': 0.257, 'Migration': 0.044, 'Other': 0.015}
Livestock/Crop/Transport train rows 67619 test rows 17162
Migration train rows 4446 test rows 994
Other train rows 1508 test rows 362
Trade/Housing/Fishery train rows 25621 test rows 6281

                                          MAE  RMSLE  R2_log  within_1_ladder_step  exact_ladder_step  AUC_loan_ge_300k
single model (4 features)          41079.665  0.503   0.527                 0.471              0.206             0.941
separate model per purpose family  41175.193  0.505   0.524                 0.472              0.211             0.940
Livestock/Crop/Transport   n=17162  MAE single     30929  separate     31089
Migration                  n=  994  MAE single    100445  separate    102122
Other                      n=  362  MAE single     58574  separate     58092
Trade/Housing/Fishery      n= 6281  MAE single     58413  separate     58115
